In [1]:
import json
import os
import numpy as np


def ket0():
    return np.array([1.0, 0.0], dtype=complex)


def ket1():
    return np.array([0.0, 1.0], dtype=complex)


def projector(v):
    return np.outer(v, v.conj())


def partial_trace_qubits_1_2(rho_4q: np.ndarray) -> np.ndarray:
    """
    Trace out qubits q1 and q2 from a 4-qubit density matrix.
    Qubit order is q0 q1 q2 q3.
    Returns reduced density matrix on q0,q3 in basis |00>,|01>,|10>,|11>.
    """
    rho = rho_4q.reshape(2, 2, 2, 2, 2, 2, 2, 2)
    # axes: bra(q0,q1,q2,q3), ket(q0,q1,q2,q3)
    rho = np.trace(rho, axis1=1, axis2=5)  # trace q1
    rho = np.trace(rho, axis1=1, axis2=4)  # trace q2 (after first trace, indices shift)
    return rho.reshape(4, 4)


def main():
    os.makedirs("results", exist_ok=True)

    # Bell state |Phi+> = (|00> + |11>)/sqrt(2)
    phi = (np.kron(ket0(), ket0()) + np.kron(ket1(), ket1())) / np.sqrt(2.0)
    bell_proj = projector(phi)

    # Initial 4-qubit state: |Phi+>_01 tensor |Phi+>_23
    psi = np.kron(phi, phi)  # basis order q0 q1 q2 q3

    # Project q1,q2 onto |Phi+>
    P12 = (
        np.kron(projector(ket0()), np.kron(bell_proj, projector(ket0())))
        + np.kron(projector(ket0()), np.kron(bell_proj, projector(ket1())))
        + np.kron(projector(ket1()), np.kron(bell_proj, projector(ket0())))
        + np.kron(projector(ket1()), np.kron(bell_proj, projector(ket1())))
    )

    psi_post_unnorm = P12 @ psi
    success_probability = float(np.vdot(psi_post_unnorm, psi_post_unnorm).real)

    if success_probability < 1e-15:
        raise RuntimeError("Postselected branch has zero probability.")

    psi_post = psi_post_unnorm / np.sqrt(success_probability)
    rho_post = projector(psi_post)

    # Reduced state on q0,q3
    rho_03 = partial_trace_qubits_1_2(rho_post)

    # Fidelity with target |Phi+> on q0,q3
    bell_fidelity = float(np.vdot(phi, rho_03 @ phi).real)
    entanglement_yield = success_probability * bell_fidelity

    result = {
        "stack": "pure_python",
        "paradigm": "DV",
        "protocol": "two_bellpair_entanglement_swapping",
        "success_probability": success_probability,
        "bell_fidelity": bell_fidelity,
        "entanglement_yield": entanglement_yield,
        "notes": "Minimal NumPy reference with analytic postselection on |Phi+>_12."
    }

    with open("results/pure_python.json", "w", encoding="utf-8") as f:
        json.dump(result, f, indent=2)

    print(json.dumps(result, indent=2))


if __name__ == "__main__":
    main()

{
  "stack": "pure_python",
  "paradigm": "DV",
  "protocol": "two_bellpair_entanglement_swapping",
  "success_probability": 0.24999999999999978,
  "bell_fidelity": 0.9999999999999998,
  "entanglement_yield": 0.24999999999999972,
  "notes": "Minimal NumPy reference with analytic postselection on |Phi+>_12."
}
